# WTI Hourly Price Data — Yahoo Finance

Downloads two years of hourly OHLCV (Open, High, Low, Close, Volume) data for WTI Crude Oil futures (`CL=F`) from Yahoo Finance via the `yfinance` library, alongside the dollar index (DXY) and VIX, and computes four derived liquidity metrics: log volume, price range, log return, and the Amihud illiquidity ratio.

To adapt this notebook to another commodity, change the `tickers` mapping (and the DB path); everything downstream keys off `data['wti']`.

Results are written to:
- **DB:** `market_context` table in `01_data/wti_thesis.db` (primary)
- **CSV backups:** `01_data/raw/price/wti_hourly_raw.csv` and `01_data/processed/market_context.csv`

EIA, Cushing, and any other columns added by downstream notebooks are preserved when re-running this notebook.

### Necessary imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3
import os

### Download hourly OHLCV for all tickers

Pulls two years of hourly bars for the three tickers at once: the WTI front-month future (`CL=F`), the dollar index (`DX-Y.NYB`), and the VIX (`^VIX`), returned in the `data` dict keyed by short name. All series are converted to UTC so they align on `datetime_hour` downstream. This is the commodity-specific step: swap the `tickers` mapping to retarget the whole pipeline.

In [10]:
tickers = {
    "CL=F": "wti",
    "DX-Y.NYB": "dxy",
    "^VIX": "vix"
}

data = {}
for ticker, name in tickers.items():
    df = yf.download(ticker, period="2y", interval="1h", progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index).tz_convert('UTC')
    df.index.name = 'datetime_hour'
    data[name] = df
    print(f"{name.upper()}: {len(df)} records | {df.index.min().date()} to {df.index.max().date()}")

WTI: 11224 records | 2024-05-12 to 2026-05-12
DXY: 11939 records | 2024-05-12 to 2026-05-12
VIX: 7114 records | 2024-05-13 to 2026-05-12


### Obtain liquidity information

The raw OHLCV columns feeding the derived metrics below:

- **Open:** price at the moment of opening the time window.
- **High:** highest price in the time window.
- **Low:** lowest price in the time window.
- **Volume:** amount of contracts in the time window.
- **Close:** price at the moment of closing the time window.

### Calculate liquidity variables derived from the OHLCV entries

Four liquidity metrics are calculated from the raw OHLCV data:

- **log_volume:** 
  - Directly measures trading activity — how many contracts changed hands
  - Log transformation handles the right-skewed distribution correctly
  - This IS liquidity in the most direct sense for futures markets
  - Standard in market microstructure literature

- **price_range:**
  - High − Low within the hour captures intraday volatility
  - Wider range = more price uncertainty = less liquidity (wider bid-ask spreads)
  - Parkinson (1980) is a well-cited estimator, defensible academically
  - Correlated with bid-ask spread which we can't get from yfinance

- **log_return:** log(Close_t / Close_{t-1}) — the hourly price return.
  - No liquidity variable, but price variable, measures how much the priced moved, not how easy it was to trade. This is not a dependent variable

- **amihud:** 
  - |log_return| / Volume = price impact per unit of volume
  - High Amihud = prices move a lot per contract traded = illiquid market
  - Amihud (2002) is arguably the most cited liquidity measure in finance literature
  - Captures the market impact dimension of liquidity


In [12]:
# Cell 4 — Compute WTI liquidity features
df_wti = data['wti'].copy()
df_wti['log_volume'] = np.log(df_wti['Volume'] + 1)
df_wti['price_range'] = df_wti['High'] - df_wti['Low']
df_wti['log_return'] = np.log(df_wti['Close'] / df_wti['Close'].shift(1))
df_wti['amihud'] = df_wti['log_return'].abs() / (df_wti['Volume'] + 1)

print("Liquidity features computed:")
print(df_wti[['log_volume', 'price_range', 'amihud']].describe().round(4))

Liquidity features computed:
Price  log_volume  price_range      amihud
count  11224.0000   11224.0000  11223.0000
mean       8.3222       0.4890      0.0001
std        2.1840       0.6176      0.0013
min        0.0000       0.0000      0.0000
25%        7.5147       0.1900      0.0000
50%        8.7252       0.3300      0.0000
75%        9.7692       0.5400      0.0000
max       13.3046      14.4000      0.1107


### Assemble and persist `market_context`

Builds the `market_context` frame: the four WTI liquidity metrics joined with the DXY and VIX closes, with VIX forward-filled up to 16 hours to cover off-market gaps. The `try/except` preserves any `cushing_inventory`, `eia_surprise`, and `is_eia_release` already written by downstream notebooks, so re-running here does not wipe them (on a fresh DB those columns start empty). Outputs are written CSV-first (`wti_hourly_raw.csv`, `market_context.csv`) as a restore point, then to the `market_context` table with `if_exists='replace'`.

In [ ]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_dxy = data['dxy'][['Close']].rename(columns={'Close': 'dxy'})
df_vix = data['vix'][['Close']].rename(columns={'Close': 'vix'})

df_market = df_wti[['log_volume', 'price_range', 'log_return', 'amihud']].copy()
df_market = df_market.join(df_dxy, how='left')
df_market = df_market.join(df_vix, how='left')
# Forward-fill VIX for off-hours gaps (max 16h to avoid stale values)
df_market['vix'] = df_market['vix'].ffill(limit=16)

# Preserve columns added by downstream notebooks so re-running this doesn't wipe them
try:
    df_existing = pd.read_sql(
        "SELECT datetime_hour, cushing_inventory, eia_surprise, is_eia_release FROM market_context",
        conn
    )
    df_existing['datetime_hour'] = pd.to_datetime(df_existing['datetime_hour'], utc=True)
    df_existing = df_existing.set_index('datetime_hour')
    df_market = df_market.join(df_existing, how='left')
    print(f"Preserved EIA + Cushing from {len(df_existing)} existing rows")
except Exception:
    df_market['cushing_inventory'] = None
    df_market['eia_surprise'] = None
    df_market['is_eia_release'] = None
    print("No existing market_context — EIA and Cushing columns initialized empty")

df_market = df_market.reset_index()
df_market['datetime_hour'] = df_market['datetime_hour'].astype(str)

# Save CSVs FIRST — if the DB write fails, these are the restore point
raw_path = "../01_data/raw/price/wti_hourly_raw.csv"
os.makedirs(os.path.dirname(os.path.abspath(raw_path)), exist_ok=True)
df_wti.reset_index().to_csv(raw_path, index=False)
print(f"Raw WTI CSV saved:     {raw_path} ({len(df_wti)} rows)")

mc_path = "../01_data/processed/market_context.csv"
df_market.to_csv(mc_path, index=False)
print(f"market_context CSV saved: {mc_path} ({len(df_market)} rows)")

# Write to DB after CSVs are safe
df_market.to_sql('market_context', conn, if_exists='replace', index=False)
print(f"market_context DB saved: {len(df_market)} rows")
print(f"DXY coverage: {df_market['dxy'].notna().sum()} / {len(df_market)}")
print(f"VIX coverage: {df_market['vix'].notna().sum()} / {len(df_market)}")

### Verify

Sanity check on the written table: the most recent rows with DXY/VIX/EIA populated, plus coverage counts for each joined column.

In [ ]:
sample = pd.read_sql("""
    SELECT datetime_hour, log_volume, dxy, vix, eia_surprise, is_eia_release
    FROM market_context
    WHERE dxy IS NOT NULL
    ORDER BY datetime_hour DESC
    LIMIT 5
""", conn)
print(sample)

counts = pd.read_sql("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN dxy IS NOT NULL THEN 1 ELSE 0 END) as dxy_count,
        SUM(CASE WHEN vix IS NOT NULL THEN 1 ELSE 0 END) as vix_count,
        SUM(CASE WHEN eia_surprise IS NOT NULL THEN 1 ELSE 0 END) as eia_count,
        SUM(CASE WHEN cushing_inventory IS NOT NULL THEN 1 ELSE 0 END) as cushing_count
    FROM market_context
""", conn)
print(counts)

conn.close()
print("Done.")